# Module 05: Multi-Agent Orchestration

**Concept brief:**

- Single agents with many tools become hard to manage and debug — every tool is a responsibility, and failures become hard to isolate
- Solution: divide and conquer — a **manager** breaks the task into sub-tasks, and **specialists** execute each one
- The manager is a pure coordinator: it has **no tools of its own**, only other agents
- Visual overview of the system we will build:

```
manager
  ├──▶ web_researcher   (DuckDuckGoSearchTool, VisitWebpageTool)
  └──▶ data_analyst     (compute_stats)
```

- **Key insight**: the specialist's `description` is the manager's routing table — the manager reads it to decide who to call and what to pass

## Setup

In [ ]:
# Install required packages
# Uncomment the line below if running in Google Colab or a fresh environment
# !uv pip install smolagents python-dotenv duckduckgo-search mlflow
# Or using pip:
# !pip install smolagents python-dotenv duckduckgo-search mlflow

In [ ]:
import os

# ----- HF_TOKEN Setup -----
# Option A: Load from .env file (local development)
# from dotenv import load_dotenv
# load_dotenv()

# Option B: Google Colab Secrets
# Uncomment the lines below when running in Google Colab.
# Go to: Colab → Secrets (🔑 icon) → Add HF_TOKEN
# from google.colab import userdata
# os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

# Option C: Set directly (not recommended for shared notebooks)
# os.environ['HF_TOKEN'] = 'hf_your_token_here'

In [ ]:
import os
import statistics
from dotenv import load_dotenv
from smolagents import CodeAgent, ToolCallingAgent, InferenceClientModel, DuckDuckGoSearchTool, VisitWebpageTool, tool

load_dotenv()

model = InferenceClientModel(
    model_id="Qwen/Qwen2.5-Coder-32B-Instruct",
    token=os.environ["HF_TOKEN"],
)
print("Ready.")

## Building Specialist Agents

We'll build two specialists:
1. **web_researcher** — searches and visits web pages
2. **data_analyst** — performs numerical analysis

Each has a clear single responsibility.

In [ ]:
web_researcher = ToolCallingAgent(
    tools=[DuckDuckGoSearchTool(), VisitWebpageTool()],
    model=model,
    max_steps=6,
    name="web_researcher",
    description=(
        "Searches the web and visits pages to retrieve factual, up-to-date information. "
        "Provide a specific research question as the task. "
        "Returns a concise summary with source URLs."
    ),
)

print(f"Specialist: {web_researcher.name}")
print(f"Description: {web_researcher.description}")

In [ ]:
@tool
def compute_stats(numbers: str) -> str:
    """
    Computes mean, median, min, max, and standard deviation for a
    comma-separated list of numbers.

    Args:
        numbers: Comma-separated numeric values, e.g. '10,20,30,40,50'.
    """
    vals = [float(x.strip()) for x in numbers.split(",")]
    mean = sum(vals) / len(vals)
    sorted_vals = sorted(vals)
    n = len(vals)
    median = sorted_vals[n // 2] if n % 2 != 0 else (sorted_vals[n // 2 - 1] + sorted_vals[n // 2]) / 2
    variance = sum((x - mean) ** 2 for x in vals) / (n - 1)
    stdev = variance ** 0.5
    return (
        f"n={n}, mean={mean:.2f}, median={median:.2f}, "
        f"min={min(vals):.2f}, max={max(vals):.2f}, stdev={stdev:.2f}"
    )

data_analyst = CodeAgent(
    tools=[compute_stats],
    model=model,
    max_steps=5,
    name="data_analyst",
    description=(
        "Performs numerical analysis and statistics on data. "
        "Provide the numbers as a comma-separated string in your task. "
        "Returns mean, median, min, max, and standard deviation."
    ),
)

print(f"Specialist: {data_analyst.name}")

## The Manager Agent

The manager has **no tools** — only `managed_agents`. It reads the task, decides which specialist to call, passes a sub-task to them, and synthesizes the results.

In [ ]:
manager = CodeAgent(
    tools=[],
    model=model,
    managed_agents=[web_researcher, data_analyst],
    max_steps=8,
)

print("Manager has", len(manager.managed_agents), "specialists")
print("Specialists:", [a.name for a in manager.managed_agents.values()])

## Running the Multi-Agent System

Let's give the manager a task that requires BOTH specialists.

In [ ]:
result = manager.run(
    "Research the average annual salary for a senior data engineer in the US in 2024. "
    "Then, given these self-reported figures from a survey: "
    "130000, 145000, 152000, 128000, 160000, 135000, 148000, "
    "compute the statistics and compare them to what you found online. "
    "Provide a brief summary with your source URL."
)
print("\nResult:")
print(result)

## Inspecting Delegation

We can inspect `manager.memory.steps` to see exactly when and how the manager delegated to specialists.

In [ ]:
print(f"Manager used {len(manager.memory.steps)} steps\n")

for i, step in enumerate(manager.memory.steps):
    step_type = type(step).__name__
    step_str = str(step)
    # Look for agent delegation calls in the step
    if "web_researcher" in step_str or "data_analyst" in step_str:
        print(f"Step {i} [{step_type}]: DELEGATION DETECTED")
        print(f"  {step_str[:300]}")
    else:
        print(f"Step {i} [{step_type}]: {step_str[:150]}")
    print()

## Design Principles

### The One-Responsibility Rule
Each specialist should do *one thing well*. A specialist that does research AND analysis AND formatting is hard to debug and easy to misuse.

### Descriptions Are the Routing Table
The manager reads specialist descriptions to decide who to call. Poor description = wrong routing. Include: what the specialist does, what input format it expects, what it returns.

### Keep Specialists Stateless
Each specialist call is independent. Don't rely on a specialist remembering context from a previous call.

### Set Appropriate `max_steps`
A web researcher needs more steps (5–8) than a calculator (2–3). Match `max_steps` to the complexity of the specialist's job.

## Exercises

In [ ]:
# TODO Exercise 1: Add a report_writer specialist
# 1. Create a third specialist agent called "report_writer" with:
#    - Tools: none (it just formats text — CodeAgent with add_base_tools=True for string ops)
#    - Description: "Formats research findings into a structured markdown report 
#      with sections: Summary, Key Findings, Data Analysis, Conclusion."
#    - max_steps: 3
#
# 2. Add it to the manager's managed_agents
#
# 3. Run: "Research the average data scientist salary in 2024, analyze these 
#    figures: 115000, 132000, 98000, 145000, 120000, and produce a 
#    formatted markdown report."
#
# Does the manager use all three specialists?

# Your code here:

In [ ]:
# TODO Exercise 2: Data pipeline multi-agent system
# Design a 2-specialist system:
#   Specialist 1 (searcher): finds a public CSV dataset URL on a topic of your choice
#     (e.g., "world population data CSV download URL")
#   Specialist 2 (analyzer): uses the CSVSummaryTool from Module 02 to analyze it
#     Hint: copy the CSVSummaryTool class definition here, or re-implement it
#
# Manager task: "Find a public CSV dataset about [your topic], 
#               download the URL and analyze its structure."
#
# Challenge: Can the searcher's output (a URL) be passed to the analyzer?
# How would you structure the manager task to make this work?

# Your code here:

## What You Built

✅ **You now know how to:**
- Build specialist agents with focused roles and clear descriptions
- Create a manager agent that delegates via `managed_agents`
- Inspect `memory.steps` to trace delegation decisions
- Apply the one-responsibility, stateless specialist design principles
- Recognize when multi-agent adds value (complex multi-step tasks) vs overhead (simple tasks)

**Key insight:** The specialist's `description` is the manager's routing table. If the manager makes bad delegation choices, improve the description first — not the manager's prompt.

## Next Module Preview

➡️ **Module 06: MLflow Observability**

Your multi-agent systems are powerful but non-deterministic — the same prompt can take different paths and produce different results. In Module 06 you'll add MLflow to record every run: which model, which tools, how many steps, how long it took. This is how you iterate on agent design scientifically.